# Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
from datetime import datetime
import itertools

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.colors as pc
import seaborn as sns

from scipy import stats
from scipy.integrate import trapezoid
from scipy.stats import gaussian_kde, mannwhitneyu, ks_2samp, pearsonr, spearmanr, kruskal
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import classification_report, confusion_matrix

from utils import *

warnings.filterwarnings('ignore')

# Funções auxiliares

Todas as funções do estudo ficam em **`utils.py`**, importado na célula acima com `from utils import *`.

| Bloco | Funções |
|---|---|
| Filtro de Hampel | `hampel`, `ResultadoHampel` |
| Carga e limpeza | `carregar_crystallizer`, `aplicar_tratamentos`, `carregar_eventos` |
| Janelas e eventos | `valores_na_janela`, `adicionar_eventos_ultrapassagem`, `separar_eventos_por_data`, `unificar_eventos` |
| Gráficos da série | `plot_crystallizer`, `plot_crystallizers`, `plot_mm_crystallizer`, `plot_mm_crystallizers`, `plot_crystallizer_unificado` |
| Estatística | `estatisticas`, `plot_violin_classes`, `plot_violin_serie_vs_ultrapassagem`, `testar_classes_por_janela`, `separabilidade_features` |
| Clusterização | `extrair_features_clustering`, `pipeline_clustering`, `rodar_clusterizacao`, `plotar_clusterizacao_pca` |
| Classificação | `extrair_features`, `dividir_dados`, `selecionar_features`, `definir_modelos_e_grids`, `avaliar_cv_com_grid`, `avaliar_teste`, `rodar_classificacao` |
| IsolationForest | `otimizar_avaliar_iforest`, `rodar_iforest` |
| Cartas de controle | `calcular_ewma`, `otimizar_ewma`, `calcular_cusum_dinamico`, `otimizar_cusum_dinamico`, `avaliar_carta_controle` |

Constantes compartilhadas (`RANDOM_STATE`, `N_SPLITS`, `TEST_SIZE`, `STATS_FUNCS`, `COLUNA_FE`) também vêm de lá.
Depois de editar o `utils.py` é preciso reiniciar o kernel para o notebook enxergar a mudança.

# Carregando dados

## Política de tratamento de dados — onde cada versão pode ser usada

O sinal que antecipa falha é justamente a leitura alta de ferro: **o outlier é o alvo**, e filtro
estatístico de outlier e detector de falha são a mesma operação com sinais trocados. Verificação
feita contra as duas leituras que a planilha de inspeção confirma como **causa** de parada de
emergência (C3 264 ppm em 23/11/2013 e C3 999 ppm em 02/08/2018, ambas com furo confirmado na
abertura) e contra o detector diário sobre as 31 falhas ancoradas:

| Tratamento | Leituras-gatilho | Amostras >10 ppm (C3) | Detector max>10 (F2) | Detector mediana≥3.5 (F2) |
|---|---|---|---|---|
| Original + `limpar_excursoes` | **mantidas** | 24 | 0.181 | 0.231 |
| Intervalo 0-10 | removidas | 0 | 0.000 | 0.233 |
| IQR | removidas | 0 (teto 4.0 ppm) | 0.000 | 0.236 |
| Hampel 15d / 90d | removidas | 0 | 0.000 | 0.194 / 0.154 |

Duas conclusões:

1. **Todo filtro estatístico deleta o alvo.** No IQR o maior valor sobrevivente é 4.0 ppm —
   abaixo do próprio limite operacional de 5 ppm: avaliar a regra vigente sobre dados IQR é
   impossível por construção.
2. **A estatística robusta substitui o filtro.** A mediana diária entrega o mesmo F2 sobre dados
   brutos e filtrados — o filtro não melhora o detector robusto e destrói o detector de cauda.

**Decisão (duas pistas):**

- **Pista A — canônica** (tabelas de evento, features, modelos, detector, baseline): sempre
  `Original` = bruto + `limpar_excursoes`. A separação erro-de-laboratório vs excursão real é
  feita por regra **causal** (corroboração na vizinhança OU parada de amostragem em seguida),
  não por distância estatística. Robustez a spike espúrio vem de features robustas (mediana,
  p75/p90) e das features relativas — não de pré-filtro.
- **Pista B — visualização e EDA distribucional**: `Intervalo 0-10`, `IQR` e `Hampel` continuam
  existindo para gráficos de série, KDE/violin e comparação descritiva (sem eles, 999 ppm
  esmaga qualquer escala). **Nunca alimentam janela de evento nem modelo.**

As células que comparam tratamentos lado a lado (estatística descritiva, effect size por
tratamento, violins por classe) foram mantidas de propósito: são a evidência desta decisão.

In [ ]:
# Parâmetros do filtro de Hampel (janela em amostras = dias * amostras por dia)
dias             = 15   # janela curta
dias_longo       = 90   # janela longa, usada para limpar a série completa
amostras_por_dia = 5    # aproximação: a base tem ~6 amostras de laboratório por dia

window_size = dias * amostras_por_dia

## Crystallizer #1

In [ ]:
base_name_crystallizer1 = "Crystallizer #1.csv"

# Linhas 23 e 2141 estão duplicadas mas não possuem amostras significativas (i.e amostras com valores muito baixos)
df_crystallizer1, df_duplicados_crystallizer1 = carregar_crystallizer(
    base_name_crystallizer1, linhas_remover=[23, 2141], limite_ppm=None
)

# limite_ppm=None desliga o corte fixo; a limpeza de excursões abaixo é que decide
# o que é erro de laboratório e o que é medição alta informativa
df_crystallizer1, df_descartes_crystallizer1 = limpar_excursoes(df_crystallizer1)

tratamentos_crystallizer1, info_crystallizer1 = aplicar_tratamentos(
    df_crystallizer1,
    dias_hampel=dias, dias_hampel_longo=dias_longo, amostras_por_dia=amostras_por_dia
)

df_crystallizer1

### Removendo outliers 0 a 10 #1

In [ ]:
df_crystallizer1_0a10 = tratamentos_crystallizer1["Intervalo 0-10"]

print(f"Amostras originais : {len(df_crystallizer1)}")
print(f"Amostras no intervalo [0, 10]: {len(df_crystallizer1_0a10)} ({len(df_crystallizer1) - len(df_crystallizer1_0a10)} removidas)")

df_crystallizer1_0a10

### Removendo outliers com IQR #1

In [ ]:
df_crystallizer1_iqr = tratamentos_crystallizer1["IQR"]
limite_inf, limite_sup = info_crystallizer1["limites_iqr"]

print(f"Limites IQR       : [{limite_inf:.3f}, {limite_sup:.3f}]")
print(f"Amostras originais: {len(df_crystallizer1)}")
print(f"Amostras removidas: {len(df_crystallizer1) - len(df_crystallizer1_iqr)}")
print(f"Amostras restantes: {len(df_crystallizer1_iqr)}")

df_crystallizer1_iqr

### Removendo outliers com Filtro de Hampel #1

In [ ]:
df_crystallizer1_hampel = tratamentos_crystallizer1["Hampel"]

print(f"Janela: {dias} dias ({dias * amostras_por_dia} amostras)")
print(f"Outliers detectados: {info_crystallizer1['outliers_hampel']}")
df_crystallizer1_hampel

#### Removendo outliers com Filtro de Hampel window_size=90 dias #1

In [ ]:
df_crystallizer1_hampel90d = tratamentos_crystallizer1["Hampel 90d"]

print(f"Janela: {dias_longo} dias ({dias_longo * amostras_por_dia} amostras)")
print(f"Outliers detectados: {info_crystallizer1['outliers_hampel_longo']}")
df_crystallizer1_hampel90d

## Crystallizer #2

In [ ]:
base_name_crystallizer2 = "Crystallizer #2.csv"

# A amostra de 20000 ppm (Labref 4027521) é removida pelo teto de implausibilidade
df_crystallizer2, df_duplicados_crystallizer2 = carregar_crystallizer(
    base_name_crystallizer2, limite_ppm=None
)

# limite_ppm=None desliga o corte fixo; a limpeza de excursões abaixo é que decide
# o que é erro de laboratório e o que é medição alta informativa
df_crystallizer2, df_descartes_crystallizer2 = limpar_excursoes(df_crystallizer2)

tratamentos_crystallizer2, info_crystallizer2 = aplicar_tratamentos(
    df_crystallizer2,
    dias_hampel=dias, dias_hampel_longo=dias_longo, amostras_por_dia=amostras_por_dia
)

df_crystallizer2

### Removendo outliers 0 a 10 #2

In [ ]:
df_crystallizer2_0a10 = tratamentos_crystallizer2["Intervalo 0-10"]

print(f"Amostras originais : {len(df_crystallizer2)}")
print(f"Amostras no intervalo [0, 10]: {len(df_crystallizer2_0a10)} ({len(df_crystallizer2) - len(df_crystallizer2_0a10)} removidas)")

df_crystallizer2_0a10

### Removendo outliers com IQR #2

In [ ]:
df_crystallizer2_iqr = tratamentos_crystallizer2["IQR"]
limite_inf, limite_sup = info_crystallizer2["limites_iqr"]

print(f"Limites IQR       : [{limite_inf:.3f}, {limite_sup:.3f}]")
print(f"Amostras originais: {len(df_crystallizer2)}")
print(f"Amostras removidas: {len(df_crystallizer2) - len(df_crystallizer2_iqr)}")
print(f"Amostras restantes: {len(df_crystallizer2_iqr)}")

df_crystallizer2_iqr

### Removendo outliers com Filtro de Hampel #2

In [ ]:
df_crystallizer2_hampel = tratamentos_crystallizer2["Hampel"]

print(f"Janela: {dias} dias ({dias * amostras_por_dia} amostras)")
print(f"Outliers detectados: {info_crystallizer2['outliers_hampel']}")
df_crystallizer2_hampel

#### Removendo outliers com Filtro de Hampel window_size=90 dias #2

In [ ]:
df_crystallizer2_hampel90d = tratamentos_crystallizer2["Hampel 90d"]

print(f"Janela: {dias_longo} dias ({dias_longo * amostras_por_dia} amostras)")
print(f"Outliers detectados: {info_crystallizer2['outliers_hampel_longo']}")
df_crystallizer2_hampel90d

## Crystallizer #3

In [ ]:
base_name_crystallizer3 = "Crystallizer #3.csv"

# Linhas 6476 e 6494 estão duplicadas mas não possuem amostras significativas
df_crystallizer3, df_duplicados_crystallizer3 = carregar_crystallizer(
    base_name_crystallizer3, linhas_remover=[6476, 6494], limite_ppm=None
)

# limite_ppm=None desliga o corte fixo; a limpeza de excursões abaixo é que decide
# o que é erro de laboratório e o que é medição alta informativa
df_crystallizer3, df_descartes_crystallizer3 = limpar_excursoes(df_crystallizer3)

tratamentos_crystallizer3, info_crystallizer3 = aplicar_tratamentos(
    df_crystallizer3,
    dias_hampel=dias, dias_hampel_longo=dias_longo, amostras_por_dia=amostras_por_dia
)

df_crystallizer3

### Removendo outliers 0 a 10 #3

In [ ]:
df_crystallizer3_0a10 = tratamentos_crystallizer3["Intervalo 0-10"]

print(f"Amostras originais : {len(df_crystallizer3)}")
print(f"Amostras no intervalo [0, 10]: {len(df_crystallizer3_0a10)} ({len(df_crystallizer3) - len(df_crystallizer3_0a10)} removidas)")

df_crystallizer3_0a10

### Removendo outliers com IQR #3

In [ ]:
df_crystallizer3_iqr = tratamentos_crystallizer3["IQR"]
limite_inf, limite_sup = info_crystallizer3["limites_iqr"]

print(f"Limites IQR       : [{limite_inf:.3f}, {limite_sup:.3f}]")
print(f"Amostras originais: {len(df_crystallizer3)}")
print(f"Amostras removidas: {len(df_crystallizer3) - len(df_crystallizer3_iqr)}")
print(f"Amostras restantes: {len(df_crystallizer3_iqr)}")

df_crystallizer3_iqr

### Removendo outliers com Filtro de Hampel #3

In [ ]:
df_crystallizer3_hampel = tratamentos_crystallizer3["Hampel"]

print(f"Janela: {dias} dias ({dias * amostras_por_dia} amostras)")
print(f"Outliers detectados: {info_crystallizer3['outliers_hampel']}")
df_crystallizer3_hampel

#### Removendo outliers com Filtro de Hampel window_size=90 dias #3

In [ ]:
df_crystallizer3_hampel90d = tratamentos_crystallizer3["Hampel 90d"]

print(f"Janela: {dias_longo} dias ({dias_longo * amostras_por_dia} amostras)")
print(f"Outliers detectados: {info_crystallizer3['outliers_hampel_longo']}")
df_crystallizer3_hampel90d

# Erro de laboratório vs medição alta informativa

O corte fixo de 100 ppm que existia antes descartava as duas leituras que a planilha de inspeção registra como **causa** de parada de emergência (264 ppm em 23/11/2013 e 999 ppm em 02/08/2018, ambas no C3, ambas com furo confirmado na abertura).

`limpar_excursoes` usa duas camadas no lugar do corte:

1. **teto de implausibilidade** (1000 ppm) — na base inteira existe uma única amostra acima disso (20000 ppm no C2); a segunda maior é 999;
2. **corroboração ou parada** — uma leitura acima de 20 ppm só é descartada se for isolada: sem outras amostras altas na vizinhança **e** sem parada de amostragem logo depois. O segundo critério é o que salva os casos graves — numa falha abrupta a operação para o reator após a primeira leitura, então a excursão não chega a aparecer em outras amostras.

In [ ]:
# Todas as amostras acima de 20 ppm e o veredito da regra
relatorio_excursoes = []
for nome, df in [("C1", df_crystallizer1), ("C2", df_crystallizer2), ("C3", df_crystallizer3)]:
    c = classificar_amostras_altas(df)
    alta = c[c["Alta"]].copy()
    alta["Crystallizer"] = nome
    relatorio_excursoes.append(alta)

relatorio_excursoes = pd.concat(relatorio_excursoes, ignore_index=True)
relatorio_excursoes[["Crystallizer", "TIMESTAMP", "Resultado de Ferro (ppm)",
                     "Corroborada", "SeguidaDeParada", "ExcursaoReal"]]

In [ ]:
# O que cada política de limpeza faria com as amostras altas
for nome, desc in [("C1", df_descartes_crystallizer1),
                   ("C2", df_descartes_crystallizer2),
                   ("C3", df_descartes_crystallizer3)]:
    print(f"{nome}: {len(desc)} amostras descartadas")
    if len(desc):
        print(desc[["TIMESTAMP", "Resultado de Ferro (ppm)", "Motivo"]].to_string(index=False))
    print()

# Carregando eventos identificados

## Deifinindo LC

In [ ]:
# LC usado para sintetizar a classe negativa (Real=0): ultrapassagens sem falha relatada.
# 10 ppm (e não os 5 ppm operacionais) torna o negativo "difícil": uma excursão clara que
# mesmo assim não terminou em falha. A variante no LC operacional de 5 ppm — o cenário real
# de alarme, com ~3x mais negativos — é construída na seção "Unificando eventos".
threshold = 10 #10 #5 #7

## Planilha de inspeção e tabela de eventos

Os eventos vêm de `data/Vitrificados do PIA - Dados de inspeção.xlsx` (uma aba por reator). Três coisas acontecem aqui:

- **reancoragem por lacuna** — quando o reator para, a amostragem para junto, e o apontamento costuma cair no meio do período sem medição. O evento é deslocado para a última medição antes da lacuna, para que a janela `[ts - dias, ts)` cubra os dados que realmente antecedem a parada;
- **parada de planta vs parada de reator** — lacunas que atingem os três reatores ao mesmo tempo são parada de planta ou de laboratório, não falha de equipamento. Eventos que caem numa dessas janelas ficam de fora do conjunto de falhas;
- **fusão de apontamentos** — a planilha registra a mesma parada em mais de uma linha; apontamentos do mesmo reator a menos de 7 dias viram um evento só.

In [ ]:
MAP_MEDICOES_BASE = {
    "C1": df_crystallizer1,
    "C2": df_crystallizer2,
    "C3": df_crystallizer3,
}

df_inspecoes_bruto = carregar_inspecoes()
paradas_de_planta  = identificar_paradas_de_planta(MAP_MEDICOES_BASE)

print(f"Paradas de planta identificadas (lacuna simultânea nos 3 reatores): {len(paradas_de_planta)}")
for inicio, fim in paradas_de_planta:
    print(f"   {inicio:%d/%m/%Y} -> {fim:%d/%m/%Y}  ({(fim - inicio).days} dias)")

In [ ]:
df_inspecoes = montar_tabela_eventos(df_inspecoes_bruto, MAP_MEDICOES_BASE, paradas_de_planta)

In [ ]:
# Os apontamentos que foram reancorados e o quanto andaram
deslocados = df_inspecoes[df_inspecoes["NoPeriodo"] & df_inspecoes["Deslocado"]]
deslocados[["Crystallizer", "Inicio", "TS_Ajustado", "DiasDeslocado", "LacunaDias",
            "LacunaDePlanta", "Selecionado", "Ocorrimento"]].sort_values(["Crystallizer", "Inicio"])

## Crystallizer #1

In [ ]:
df_eventos_crystallizer1 = eventos_para_notebook(df_inspecoes, "C1")

print(f"Falhas de reator no C1: {len(df_eventos_crystallizer1)}")
df_eventos_crystallizer1

### Adicionando momentos em que houve ultrapassagem LC
Amostras acima do limite mas que não foram indicadas como problema, a ideia é que a partir de uma ultrapassagem do limiar seja analisada uma janela anterior a essa ultrapassagem para verificar se há de fato um problema no reator

In [ ]:
DIAS_BASELINE = 15

# Atenção: esta célula acrescenta eventos ao df existente — reexecutá-la sem recarregar
# a célula anterior duplica os eventos Real=0
df_eventos_crystallizer1 = adicionar_eventos_ultrapassagem(
    df_crystallizer1, df_eventos_crystallizer1, threshold, dias_baseline=DIAS_BASELINE
)

df_eventos_crystallizer1

### Violin Plot - Testes com diferentes hiperparâmetros
Limpar serie toda com hampel = 90 dias de janela

Ultrapassagem de 10/7 ppm, com hampel de 15 dias

> **Decisão:** a comparação com uma curva normal sintética que existia aqui foi removida.
> A série tem assimetria forte e cauda pesada, então uma normal ajustada por média/desvio
> não é uma referência válida — a referência honesta é a própria série limpa com Hampel 90d.

In [ ]:
# JANELAS = [15, 12, 9, 6, 3]
JANELAS = [15, 7]

metodos = [
    # {"titulo": "Original",       "df": df_crystallizer1},
    # {"titulo": "Intervalo 0-10", "df": df_crystallizer1_0a10},
    # {"titulo": "IQR",            "df": df_crystallizer1_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer1_hampel},
]

# Referência: série inteira limpa com Hampel de 90 dias
valores_serie = df_crystallizer1_hampel90d["Resultado de Ferro (ppm)"].dropna().tolist()

fig = plot_violin_serie_vs_ultrapassagem(
    df_eventos_crystallizer1, metodos, JANELAS,
    titulo=(f"Violin Plot: Série Completa vs Janelas de Ultrapassagem do LC ({threshold}ppm) — Crystallizer #1"
            "<br> <sup>Esquerda (azul): distribuição Hampel 90d  |  "
            "Direita (vermelho): distribuição nos N dias antes de qualquer ultrapassagem</sup>"),
    valores_referencia=valores_serie,
)
fig.show()

In [ ]:
# fig.write_html("ViolinPlot_Cristallyzer#1.html")

### Separando eventos até 01/04/2020 e após 01/04/2020

In [ ]:
df_eventos_crystallizer1_filtrado_until2020, df_eventos_crystallizer1_filtrado_after2020 = \
    separar_eventos_por_data(df_eventos_crystallizer1, "2020-04-01")

print(f"Até 01/04/2020 : {len(df_eventos_crystallizer1_filtrado_until2020)} eventos")
print(f"Após 01/04/2020: {len(df_eventos_crystallizer1_filtrado_after2020)} eventos")

df_eventos_crystallizer1_filtrado_until2020.head()

## Crystallizer #2

In [ ]:
df_eventos_crystallizer2 = eventos_para_notebook(df_inspecoes, "C2")

print(f"Falhas de reator no C2: {len(df_eventos_crystallizer2)}")
df_eventos_crystallizer2

### Adicionando momentos em que houve ultrapassagem do limite de 5ppm
Amostras acima do limite mas que não foram indicadas como problema, a ideia é que a partir de uma ultrapassagem do limiar seja analisada uma janela anterior a essa ultrapassagem para verificar se há de fato um problema no reator

In [ ]:
DIAS_BASELINE = 15

# Atenção: esta célula acrescenta eventos ao df existente — reexecutá-la sem recarregar
# a célula anterior duplica os eventos Real=0
df_eventos_crystallizer2 = adicionar_eventos_ultrapassagem(
    df_crystallizer2, df_eventos_crystallizer2, threshold, dias_baseline=DIAS_BASELINE
)

df_eventos_crystallizer2

### Violin Plot - Testes com diferentes hiperparâmetros
Limpar serie toda com hampel = 90 dias de janela

Ultrapassagem de 10/7 ppm, com hampel de 15 dias

> **Decisão:** a comparação com uma curva normal sintética que existia aqui foi removida.
> A série tem assimetria forte e cauda pesada, então uma normal ajustada por média/desvio
> não é uma referência válida — a referência honesta é a própria série limpa com Hampel 90d.

In [ ]:
# JANELAS = [15, 12, 9, 6, 3]
JANELAS = [15, 7]

metodos = [
    # {"titulo": "Original",       "df": df_crystallizer2},
    # {"titulo": "Intervalo 0-10", "df": df_crystallizer2_0a10},
    # {"titulo": "IQR",            "df": df_crystallizer2_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer2_hampel},
]

# Referência: série inteira limpa com Hampel de 90 dias
valores_serie = df_crystallizer2_hampel90d["Resultado de Ferro (ppm)"].dropna().tolist()

fig = plot_violin_serie_vs_ultrapassagem(
    df_eventos_crystallizer2, metodos, JANELAS,
    titulo=(f"Violin Plot: Série Completa vs Janelas de Ultrapassagem do LC ({threshold}ppm) — Crystallizer #2"
            "<br> <sup>Esquerda (azul): distribuição Hampel 90d  |  "
            "Direita (vermelho): distribuição nos N dias antes de qualquer ultrapassagem</sup>"),
    valores_referencia=valores_serie,
)
fig.show()

In [ ]:
# fig.write_html("ViolinPlot_Cristallyzer#2.html")

### Separando eventos até 01/04/2020 e após 01/04/2020

In [ ]:
df_eventos_crystallizer2_filtrado_until2020, df_eventos_crystallizer2_filtrado_after2020 = \
    separar_eventos_por_data(df_eventos_crystallizer2, "2020-04-01")

print(f"Até 01/04/2020 : {len(df_eventos_crystallizer2_filtrado_until2020)} eventos")
print(f"Após 01/04/2020: {len(df_eventos_crystallizer2_filtrado_after2020)} eventos")

df_eventos_crystallizer2_filtrado_until2020.head()

## Crystallizer #3

In [ ]:
df_eventos_crystallizer3 = eventos_para_notebook(df_inspecoes, "C3")

print(f"Falhas de reator no C3: {len(df_eventos_crystallizer3)}")
df_eventos_crystallizer3

### Adicionando momentos em que houve ultrapassagem do limite de 5ppm
Amostras acima do limite mas que não foram indicadas como problema, a ideia é que a partir de uma ultrapassagem do limiar seja analisada uma janela anterior a essa ultrapassagem para verificar se há de fato um problema no reator

In [ ]:
DIAS_BASELINE = 15

# Atenção: esta célula acrescenta eventos ao df existente — reexecutá-la sem recarregar
# a célula anterior duplica os eventos Real=0
df_eventos_crystallizer3 = adicionar_eventos_ultrapassagem(
    df_crystallizer3, df_eventos_crystallizer3, threshold, dias_baseline=DIAS_BASELINE
)

df_eventos_crystallizer3

### Violin Plot - Testes com diferentes hiperparâmetros
Limpar serie toda com hampel = 90 dias de janela

Ultrapassagem de 10/7 ppm, com hampel de 15 dias

> **Decisão:** a comparação com uma curva normal sintética que existia aqui foi removida.
> A série tem assimetria forte e cauda pesada, então uma normal ajustada por média/desvio
> não é uma referência válida — a referência honesta é a própria série limpa com Hampel 90d.

In [ ]:
# JANELAS = [15, 12, 9, 6, 3]
JANELAS = [15, 7]

metodos = [
    # {"titulo": "Original",       "df": df_crystallizer3},
    # {"titulo": "Intervalo 0-10", "df": df_crystallizer3_0a10},
    # {"titulo": "IQR",            "df": df_crystallizer3_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer3_hampel},
]

# Referência: série inteira limpa com Hampel de 90 dias
valores_serie = df_crystallizer3_hampel90d["Resultado de Ferro (ppm)"].dropna().tolist()

fig = plot_violin_serie_vs_ultrapassagem(
    df_eventos_crystallizer3, metodos, JANELAS,
    titulo=(f"Violin Plot: Série Completa vs Janelas de Ultrapassagem do LC ({threshold}ppm) — Crystallizer #3"
            "<br> <sup>Esquerda (azul): distribuição Hampel 90d  |  "
            "Direita (vermelho): distribuição nos N dias antes de qualquer ultrapassagem</sup>"),
    valores_referencia=valores_serie,
)
fig.show()

In [ ]:
# fig.write_html("ViolinPlot_Cristallyzer#3.html")

### Separando eventos até 01/04/2020 e após 01/04/2020

In [ ]:
df_eventos_crystallizer3_filtrado_until2020, df_eventos_crystallizer3_filtrado_after2020 = \
    separar_eventos_por_data(df_eventos_crystallizer3, "2020-04-01")

print(f"Até 01/04/2020 : {len(df_eventos_crystallizer3_filtrado_until2020)} eventos")
print(f"Após 01/04/2020: {len(df_eventos_crystallizer3_filtrado_after2020)} eventos")

df_eventos_crystallizer3_filtrado_until2020.head()

# Baseline: o que a regra vigente entrega

A Etapa 2 tem como objetivo superar o modelo univariado em uso (patamar fixo de ferro). Para afirmar que algo o supera é preciso primeiro medi-lo — e medir contra a **taxa base**, não contra zero.

O lift compara a fração de janelas que antecedem falha atendendo a um critério com a mesma fração em janelas sorteadas ao acaso. Lift próximo de 1 significa que o critério dispara tanto antes de falha quanto em qualquer outro momento — ou seja, não informa nada.

In [ ]:
MAP_EVENTOS_FALHA = {
    "C1": df_eventos_crystallizer1[df_eventos_crystallizer1["Real"] == 1],
    "C2": df_eventos_crystallizer2[df_eventos_crystallizer2["Real"] == 1],
    "C3": df_eventos_crystallizer3[df_eventos_crystallizer3["Real"] == 1],
}

def formatar_lift(df):
    saida = df.copy()
    saida["nas_falhas"] = (100 * saida["nas_falhas"]).round(1).astype(str) + "%"
    saida["taxa_base"]  = (100 * saida["taxa_base"]).round(1).astype(str) + "%"
    saida["lift"]       = saida["lift"].round(2).astype(str) + "x"
    return saida

df_lift = avaliar_baseline_lift(MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA, janela_dias=15)
formatar_lift(df_lift)

In [ ]:
# Mesma comparação com janela de 60 dias — o limite de 5 ppm fica ABAIXO da taxa base
df_lift_60 = avaliar_baseline_lift(MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA, janela_dias=60)
formatar_lift(df_lift_60)

## Detector: precisão, recall e F2 da regra vigente e das alternativas

Alarme = estatística diária acima do limite; alarmes a menos de 15 dias contam como um só; acerto se a falha ocorre em até 15 dias depois do alarme.

In [ ]:
configuracoes = [
    ("max",    5,   1),   # regra vigente
    ("max",    7,   1),
    ("max",    10,  1),
    ("max",    20,  1),
    ("median", 3.0, 1),
    ("median", 3.5, 1),
    ("median", 4.0, 1),
    ("median", 3.5, 2),   # exigindo 2 dias seguidos — mostra que o sinal é impulso, não rampa
]

df_detector = pd.DataFrame([
    avaliar_detector_diario(MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA,
                            estatistica=est, limite=lim, dias_seguidos=dias)
    for est, lim, dias in configuracoes
]).sort_values("F2", ascending=False)

df_detector

# Plotando Gráfico das medições
Linhas vermelhas = Eventos relatados  
Linhas azuis = Eventos de ultapassagem sem relatos

## Crystallizer #1

In [ ]:
# Sem eventos falsos
fig_crystallizer1 = plot_crystallizer(
    df_crystallizer1, df_eventos_crystallizer1,
    titulo="Fe (ppm) - Crystallizer #1",
    mostrar_falsos=False
)
fig_crystallizer1.show()

In [ ]:
# fig_crystallizer1.write_html("Crystallizer#1.html")

### Crystallizer #1 - Outliers 0 a 10

In [ ]:
# Sem eventos falsos
fig_crystallizer1_0a10 = plot_crystallizer(
    df_crystallizer1_0a10, df_eventos_crystallizer1,
    titulo="Fe (ppm) - Crystallizer #1 0a10",
    mostrar_falsos=False
)
fig_crystallizer1_0a10.show()

### Crystallizer #1 - Outliers IQR

In [ ]:
# Sem eventos falsos
fig_crystallizer1_iqr = plot_crystallizer(
    df_crystallizer1_iqr, df_eventos_crystallizer1,
    titulo="Fe (ppm) - Crystallizer #1 IQR",
    mostrar_falsos=False
)
fig_crystallizer1_iqr.show()

### Crystallizer #1 - Filtro de Hampel

In [ ]:
# Sem eventos falsos
fig_crystallizer1_hampel = plot_crystallizer(
    df_crystallizer1_hampel, df_eventos_crystallizer1,
    titulo="Fe (ppm) - Crystallizer #1 Hampel",
    mostrar_falsos=False
)
fig_crystallizer1_hampel.show()

## Crystallizer #2

In [ ]:
# Sem eventos falsos
fig_crystallizer2 = plot_crystallizer(
    df_crystallizer2, df_eventos_crystallizer2,
    titulo="Fe (ppm) - Crystallizer #2",
    mostrar_falsos=False
)
fig_crystallizer2.show()

In [ ]:
# fig_crystallizer2.write_html("Crystallizer#2.html")

### Crystallizer #2 - Outliers 0 a 10

In [ ]:
# Sem eventos falsos
fig_crystallizer2_0a10 = plot_crystallizer(
    df_crystallizer2_0a10, df_eventos_crystallizer2,
    titulo="Fe (ppm) - Crystallizer #2 0a10",
    mostrar_falsos=False
)
fig_crystallizer2_0a10.show()

### Crystallizer #2 - Outliers IQR

In [ ]:
# Sem eventos falsos
fig_crystallizer2_iqr = plot_crystallizer(
    df_crystallizer2_iqr, df_eventos_crystallizer2,
    titulo="Fe (ppm) - Crystallizer #2 IQR",
    mostrar_falsos=False
)
fig_crystallizer2_iqr.show()

### Crystallizer #2 - Filtro de Hampel

In [ ]:
# Sem eventos falsos
fig_crystallizer2_hampel = plot_crystallizer(
    df_crystallizer2_hampel, df_eventos_crystallizer2,
    titulo="Fe (ppm) - Crystallizer #2 Hampel",
    mostrar_falsos=False
)
fig_crystallizer2_hampel.show()

## Crystallizer #3

In [ ]:
# Sem eventos falsos
fig_crystallizer3 = plot_crystallizer(
    df_crystallizer3, df_eventos_crystallizer3,
    titulo="Fe (ppm) - Crystallizer #3",
    mostrar_falsos=False
)
fig_crystallizer3.show()

In [ ]:
# fig_crystallizer3.write_html("Crystallizer#3.html")

### Crystallizer #3 - Outliers 0 a 10

In [ ]:
# Sem eventos falsos
fig_crystallizer3_0a10 = plot_crystallizer(
    df_crystallizer3_0a10, df_eventos_crystallizer3,
    titulo="Fe (ppm) - Crystallizer #3 0a10",
    mostrar_falsos=False
)
fig_crystallizer3_0a10.show()

### Crystallizer #3 - Outliers IQR

In [ ]:
# Sem eventos falsos
fig_crystallizer3_iqr = plot_crystallizer(
    df_crystallizer3_iqr, df_eventos_crystallizer3,
    titulo="Fe (ppm) - Crystallizer #3 IQR",
    mostrar_falsos=False
)
fig_crystallizer3_iqr.show()

### Crystallizer #3 - Filtro de Hampel

In [ ]:
# Sem eventos falsos
fig_crystallizer3_hampel = plot_crystallizer(
    df_crystallizer3_hampel, df_eventos_crystallizer3,
    titulo="Fe (ppm) - Crystallizer #3 Hampel",
    mostrar_falsos=False
)
fig_crystallizer3_hampel.show()

## Crystallizer #1#2#3

In [ ]:
crystallizers = [
    {"df": df_crystallizer1, "df_eventos": df_eventos_crystallizer1, "nome": "Crystallizer #1", "cor": "black"},
    {"df": df_crystallizer2, "df_eventos": df_eventos_crystallizer2, "nome": "Crystallizer #2", "cor": "steelblue"},
    {"df": df_crystallizer3, "df_eventos": df_eventos_crystallizer3, "nome": "Crystallizer #3", "cor": "green"},
]

fig = plot_crystallizers(crystallizers, "Fe (ppm) - Crystallizers #1, #2 e #3", mostrar_falsos=False)
fig.show()

### Outliers 0 a 10

In [ ]:
crystallizers = [
    {"df": df_crystallizer1_0a10, "df_eventos": df_eventos_crystallizer1, "nome": "Crystallizer #1", "cor": "black"},
    {"df": df_crystallizer2_0a10, "df_eventos": df_eventos_crystallizer2, "nome": "Crystallizer #2", "cor": "steelblue"},
    {"df": df_crystallizer3_0a10, "df_eventos": df_eventos_crystallizer3, "nome": "Crystallizer #3", "cor": "green"},
]

fig = plot_crystallizers(crystallizers, "Fe (ppm) - Crystallizers #1, #2 e #3 0a10", mostrar_falsos=False)
fig.show()

### Outliers IQR

In [ ]:
crystallizers = [
    {"df": df_crystallizer1_iqr, "df_eventos": df_eventos_crystallizer1, "nome": "Crystallizer #1", "cor": "black"},
    {"df": df_crystallizer2_iqr, "df_eventos": df_eventos_crystallizer2, "nome": "Crystallizer #2", "cor": "steelblue"},
    {"df": df_crystallizer3_iqr, "df_eventos": df_eventos_crystallizer3, "nome": "Crystallizer #3", "cor": "green"},
]

fig = plot_crystallizers(crystallizers, "Fe (ppm) - Crystallizers #1, #2 e #3 IQR", mostrar_falsos=False)
fig.show()

### Outliers Hampel

In [ ]:
crystallizers = [
    {"df": df_crystallizer1_hampel, "df_eventos": df_eventos_crystallizer1, "nome": "Crystallizer #1", "cor": "black"},
    {"df": df_crystallizer2_hampel, "df_eventos": df_eventos_crystallizer2, "nome": "Crystallizer #2", "cor": "steelblue"},
    {"df": df_crystallizer3_hampel, "df_eventos": df_eventos_crystallizer3, "nome": "Crystallizer #3", "cor": "green"},
]

fig = plot_crystallizers(crystallizers, "Fe (ppm) - Crystallizers #1, #2 e #3 Hampel", mostrar_falsos=False)
fig.show()

## Gráfico com Média Móvel
Verificando se há tendência clara nos dados

## MM Crystallizer #1

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer1, df_eventos_crystallizer1,
    titulo="Fe (ppm) MM - Crystallizer #1",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #1 - Outliers 0 a 10

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer1_0a10, df_eventos_crystallizer1,
    titulo="Fe (ppm) MM - Crystallizer #1 0a10",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #1 - IQR

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer1_iqr, df_eventos_crystallizer1,
    titulo="Fe (ppm) MM - Crystallizer #1 IQR",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #1 - Filtro de Hampel

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer1_hampel, df_eventos_crystallizer1,
    titulo="Fe (ppm) MM - Crystallizer #1 Hampel",
    mostrar_falsos=False
)
fig.show()

## MM Crystallizer #2

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer2, df_eventos_crystallizer2,
    titulo="Fe (ppm) MM - Crystallizer #2",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #2 - Outliers 0 a 10

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer2_0a10, df_eventos_crystallizer2,
    titulo="Fe (ppm) MM - Crystallizer #2 0a10",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #2 - IQR

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer2_iqr, df_eventos_crystallizer2,
    titulo="Fe (ppm) MM - Crystallizer #2 IQR",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #2 - Filtro de Hampel

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer2_hampel, df_eventos_crystallizer2,
    titulo="Fe (ppm) MM - Crystallizer #2 Hampel",
    mostrar_falsos=False
)
fig.show()

## MM Crystallizer #3

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer3, df_eventos_crystallizer3,
    titulo="Fe (ppm) MM - Crystallizer #3",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #3 - Outliers 0 a 10

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer3_0a10, df_eventos_crystallizer3,
    titulo="Fe (ppm) MM - Crystallizer #3 0a10",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #3 - IQR

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer3_iqr, df_eventos_crystallizer3,
    titulo="Fe (ppm) MM - Crystallizer #3 IQR",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #3 - Filtro de Hampel

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer3_hampel, df_eventos_crystallizer3,
    titulo="Fe (ppm) MM - Crystallizer #3 Hampel",
    mostrar_falsos=False
)
fig.show()

## MM Crystallizer #1#2#3

In [ ]:
crystallizers = [
    {"df": df_crystallizer1, "df_eventos": df_eventos_crystallizer1, "nome": "Crystallizer #1"},
    {"df": df_crystallizer2, "df_eventos": df_eventos_crystallizer2, "nome": "Crystallizer #2"},
    {"df": df_crystallizer3, "df_eventos": df_eventos_crystallizer3, "nome": "Crystallizer #3"},
]

# Sem eventos falsos
fig = plot_mm_crystallizers(crystallizers, num_dias=7, mostrar_falsos=False)
fig.show()

# Estatísticas descritivas de toda série de concentração de Fe

In [ ]:
# Coluna separadora vazia
separador = pd.Series({k: "" for k in ["Contagem","Média","Mediana","Desvio Padrão","Variância",
                                        "Mínimo","Máximo","Amplitude","Q1 (25%)","Q3 (75%)","IQR",
                                        "Assimetria","Curtose"]})

c1 = pd.concat([
    estatisticas(df_crystallizer1["Resultado de Ferro (ppm)"].dropna(),       "C1 Original"),
    estatisticas(df_crystallizer1_0a10["Resultado de Ferro (ppm)"].dropna(),  "C1 Intervalo 0-10"),
    estatisticas(df_crystallizer1_iqr["Resultado de Ferro (ppm)"].dropna(),   "C1 IQR"),
    estatisticas(df_crystallizer1_hampel["Resultado de Ferro (ppm)"].dropna(),"C1 Hampel"),
], axis=1)

c2 = pd.concat([
    estatisticas(df_crystallizer2["Resultado de Ferro (ppm)"].dropna(),       "C2 Original"),
    estatisticas(df_crystallizer2_0a10["Resultado de Ferro (ppm)"].dropna(),  "C2 Intervalo 0-10"),
    estatisticas(df_crystallizer2_iqr["Resultado de Ferro (ppm)"].dropna(),   "C2 IQR"),
    estatisticas(df_crystallizer2_hampel["Resultado de Ferro (ppm)"].dropna(),"C2 Hampel"),
], axis=1)

c3 = pd.concat([
    estatisticas(df_crystallizer3["Resultado de Ferro (ppm)"].dropna(),       "C3 Original"),
    estatisticas(df_crystallizer3_0a10["Resultado de Ferro (ppm)"].dropna(),  "C3 Intervalo 0-10"),
    estatisticas(df_crystallizer3_iqr["Resultado de Ferro (ppm)"].dropna(),   "C3 IQR"),
    estatisticas(df_crystallizer3_hampel["Resultado de Ferro (ppm)"].dropna(),"C3 Hampel"),
], axis=1)

sep = separador.rename("│")

df_comparativo = pd.concat([c1, sep, c2, sep.rename("│"), c3], axis=1).round(4)

# Corrige as colunas separadoras que ficaram com float após o round
df_comparativo["│"]  = ""
df_comparativo["│"] = ""

df_comparativo

In [ ]:
metodos = [
    {
        "titulo": "Original",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "Intervalo 0-10",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_0a10["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_0a10["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_0a10["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "IQR",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_iqr["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_iqr["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_iqr["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "Hampel",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_hampel["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_hampel["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_hampel["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
]

CORES = {"C1": "#1f77b4", "C2": "#2ca02c", "C3": "#ff7f0e"}

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=2,
    subplot_titles=[
        titulo
        for m in metodos
        for titulo in [f"KDE — {m['titulo']}", f"Violin Plot — {m['titulo']}"]
    ]
)

for row_idx, metodo in enumerate(metodos, start=1):
    for c in metodo["series"]:
        serie = c["serie"]
        nome  = c["nome"]
        cor   = CORES[nome]

        # KDE na escala de densidade natural (área sob a curva = 1)
        kde = gaussian_kde(serie)
        x_range = np.linspace(serie.min(), serie.max(), 1000)
        y_kde = kde(x_range)
        y_kde = y_kde / trapezoid(y_kde, x_range)  # normaliza

        fig.add_trace(go.Scatter(
            x=x_range,
            y=y_kde,
            mode='lines',
            line=dict(color=cor, width=2),
            fill='tozeroy',
            fillcolor=hex_to_rgba(cor, alpha=0.15),
            name=nome,
            legendgroup=nome,
            showlegend=(row_idx == 1)
        ), row=row_idx, col=1)

        # Violin
        fig.add_trace(go.Violin(
            y=serie,
            name=nome,
            marker_color=cor,
            fillcolor=hex_to_rgba(cor, alpha=0.4),
            box_visible=True,
            meanline_visible=True,
            legendgroup=nome,
            showlegend=False
        ), row=row_idx, col=2)

    fig.update_xaxes(title_text="Resultado de Ferro (ppm)", row=row_idx, col=1)
    fig.update_yaxes(title_text="Densidade", row=row_idx, col=1)
    fig.update_yaxes(title_text="ppm", row=row_idx, col=2)

fig.update_layout(
    height=500 * n_metodos,
    template='plotly_white',
    title="Análise Descritiva Comparativa — Resultado de Ferro (ppm)",
)
fig.show()

### Teste de Kruskal-Wallis com Effect Size (η²)

Para amostras grandes (n ~ 30.000), testes estatísticos como o Kruskal-Wallis tendem a rejeitar a hipótese nula mesmo com diferenças praticamente irrelevantes. Por isso o p-value é complementado pelo **eta-quadrado (η²)** que mede a proporção da variação total explicada pelo agrupamento por crystallizer.

| η²        | Interpretação                                      |
|-----------|----------------------------------------------------|
| < 0.01    | Efeito negligenciável — unificação justificada     |
| 0.01–0.06 | Efeito pequeno — unificação provavelmente aceitável|
| 0.06–0.14 | Efeito médio — avaliar com cautela                 |
| > 0.14    | Efeito grande — distribuições substancialmente diferentes |

A decisão de unificar os dados dos três crystallizers deve considerar em conjunto o η², o p-value e a inspeção visual das curvas KDE e violin plots gerados.

In [ ]:
metodos = [
    {
        "titulo": "Original",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "Intervalo 0-10",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_0a10["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_0a10["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_0a10["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "IQR",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_iqr["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_iqr["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_iqr["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "Hampel",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_hampel["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_hampel["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_hampel["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
]

for metodo in metodos:
    series = [c["serie"].values for c in metodo["series"]]
    nomes  = [c["nome"] for c in metodo["series"]]
    n_total = sum(len(s) for s in series)

    stat_kw, p_kw = kruskal(*series)

    # Eta-quadrado: mede o quanto da variação total é explicada pelo grupo
    # 0.01 = pequeno, 0.06 = médio, 0.14 = grande
    eta2 = (stat_kw - len(series) + 1) / (n_total - len(series))

    print(f"\nMétodo: {metodo['titulo']}")
    print(f"  H = {stat_kw:.4f}  |  p = {p_kw:.6f}  |  η² = {eta2:.4f}")
    if eta2 < 0.01:
        print("  → Efeito negligenciável — unificação justificada mesmo com p < 0.05")
    elif eta2 < 0.06:
        print("  → Efeito pequeno — unificação provavelmente aceitável")
    elif eta2 < 0.14:
        print("  → Efeito médio — avaliar com cautela")
    else:
        print("  → Efeito grande — distribuições substancialmente diferentes")

### Teste de normalidade Q-Q Plot

In [ ]:
# Dataset: Original — testar normalidade sobre a série IQR seria circular (o IQR amputa as
# caudas e a conclusão sai enviesada para "normal"). Ver "Política de tratamento de dados".
df = df_crystallizer1.copy()
# df = df_crystallizer1_0a10.copy()
# df = df_crystallizer1_iqr.copy()
# df = df_crystallizer1_hampel.copy()

serie = df["Resultado de Ferro (ppm)"].dropna()

# Visualização
fig_normalidade_crystallizer1 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Q-Q Plot", "Histograma + Distribuição Normal Teórica"]
)

# Q-Q Plot
qq = stats.probplot(serie, dist="norm")
qq_x = [qq[0][0][0], qq[0][0][-1]]
qq_y = [qq[1][1] + qq[1][0] * qq[0][0][0],
        qq[1][1] + qq[1][0] * qq[0][0][-1]]

fig_normalidade_crystallizer1.add_trace(go.Scatter(
    x=qq[0][0], y=qq[0][1],
    mode='markers',
    marker=dict(color='steelblue', size=4, opacity=0.5),
    name='Quantis observados'
), row=1, col=1)

fig_normalidade_crystallizer1.add_trace(go.Scatter(
    x=qq_x, y=qq_y,
    mode='lines',
    line=dict(color='red', width=2),
    name='Linha normal teórica'
), row=1, col=1)

# Histograma + curva normal teórica
x_range = np.linspace(serie.min(), serie.max(), 300)
y_normal = stats.norm.pdf(x_range, serie.mean(), serie.std())
y_normal_scaled = y_normal * len(serie) * (serie.max() - serie.min()) / 50

fig_normalidade_crystallizer1.add_trace(go.Histogram(
    x=serie, nbinsx=50,
    marker_color='steelblue', opacity=0.6,
    name='Dados observados'
), row=1, col=2)

fig_normalidade_crystallizer1.add_trace(go.Scatter(
    x=x_range, y=y_normal_scaled,
    mode='lines',
    line=dict(color='red', width=2),
    name='Normal teórica'
), row=1, col=2)

fig_normalidade_crystallizer1.update_xaxes(title_text="Quantis teóricos", row=1, col=1)
fig_normalidade_crystallizer1.update_yaxes(title_text="Quantis observados", row=1, col=1)
fig_normalidade_crystallizer1.update_xaxes(title_text="Resultado de Ferro (ppm)", row=1, col=2)
fig_normalidade_crystallizer1.update_yaxes(title_text="Contagem", row=1, col=2)

fig_normalidade_crystallizer1.update_layout(
    height=500,
    template='plotly_white',
    title="Análise de Normalidade — Resultado de Ferro (ppm) Cristallyzer #1"
)
fig_normalidade_crystallizer1.show()

# Unificando bases de dados

## Unificando dados

In [ ]:
# Original
df_crystallizer123 = pd.concat([
    df_crystallizer1.assign(Crystallizer='C1'),
    df_crystallizer2.assign(Crystallizer='C2'),
    df_crystallizer3.assign(Crystallizer='C3'),
], ignore_index=True).sort_values('TIMESTAMP')
print(f"Originais: {df_crystallizer123.shape}")

# Intervalo 0-10
df_crystallizer123_0a10 = pd.concat([
    df_crystallizer1_0a10.assign(Crystallizer='C1'),
    df_crystallizer2_0a10.assign(Crystallizer='C2'),
    df_crystallizer3_0a10.assign(Crystallizer='C3'),
], ignore_index=True).sort_values('TIMESTAMP')
print(f"Intervalo 0-10: {df_crystallizer123_0a10.shape}")

# IQR
df_crystallizer123_iqr = pd.concat([
    df_crystallizer1_iqr.assign(Crystallizer='C1'),
    df_crystallizer2_iqr.assign(Crystallizer='C2'),
    df_crystallizer3_iqr.assign(Crystallizer='C3'),
], ignore_index=True).sort_values('TIMESTAMP')
print(f"IQR: {df_crystallizer123_iqr.shape}")

# Hampel
df_crystallizer123_hampel = pd.concat([
    df_crystallizer1_hampel.assign(Crystallizer='C1'),
    df_crystallizer2_hampel.assign(Crystallizer='C2'),
    df_crystallizer3_hampel.assign(Crystallizer='C3'),
], ignore_index=True).sort_values('TIMESTAMP')
print(f"Hampel: {df_crystallizer123_hampel.shape}")

## Unificando eventos

In [ ]:
# Eventos unificados — igual para todos os tratamentos.
# A deduplicação cross-reator (remoção de Real=0 perto de qualquer Real=1 e espaçamento
# mínimo entre Real=0) vive em utils.unificar_eventos, usada também pela variante de
# 5 ppm abaixo — as duas tabelas passam exatamente pela mesma regra.
df_eventos_crystallizer123 = unificar_eventos({
    "C1": df_eventos_crystallizer1,
    "C2": df_eventos_crystallizer2,
    "C3": df_eventos_crystallizer3,
}, intervalo_min_dias=15)

## Variante: classe negativa no LC operacional (5 ppm)

O `threshold = 10` gera negativos "difíceis" (excursões claras sem falha), mas a regra vigente
e o custo da condenação indevida vivem em **5 ppm**. Para a Etapa 2 responder "reduzimos os
falsos positivos da regra vigente?", a classe negativa também precisa existir no limiar em que
a planta opera — o que multiplica os negativos e torna o problema mais realista.

As duas tabelas seguem em paralelo: `df_eventos_*` (LC 10 ppm) e `df_eventos_*_lc5` (LC 5 ppm).
Qualquer resultado supervisionado deve ser reportado nas duas.

In [ ]:
LC_OPERACIONAL = 5

# Reconstrói a parte Real=1 direto da planilha de inspeção (eventos_para_notebook) em vez de
# reaproveitar df_eventos_crystallizerN — evita herdar os Real=0 de LC 10 e o risco de
# duplicação por reexecução.
df_eventos_crystallizer1_lc5 = adicionar_eventos_ultrapassagem(
    df_crystallizer1, eventos_para_notebook(df_inspecoes, "C1"),
    LC_OPERACIONAL, dias_baseline=DIAS_BASELINE)
df_eventos_crystallizer2_lc5 = adicionar_eventos_ultrapassagem(
    df_crystallizer2, eventos_para_notebook(df_inspecoes, "C2"),
    LC_OPERACIONAL, dias_baseline=DIAS_BASELINE)
df_eventos_crystallizer3_lc5 = adicionar_eventos_ultrapassagem(
    df_crystallizer3, eventos_para_notebook(df_inspecoes, "C3"),
    LC_OPERACIONAL, dias_baseline=DIAS_BASELINE)

df_eventos_crystallizer123_lc5 = unificar_eventos({
    "C1": df_eventos_crystallizer1_lc5,
    "C2": df_eventos_crystallizer2_lc5,
    "C3": df_eventos_crystallizer3_lc5,
}, intervalo_min_dias=15)

# Comparação das duas classes negativas
comparacao = pd.DataFrame({
    "LC 10 ppm": df_eventos_crystallizer123["Real"].value_counts(),
    "LC 5 ppm (operacional)": df_eventos_crystallizer123_lc5["Real"].value_counts(),
}).rename(index={1: "Real=1 (falhas)", 0: "Real=0 (ultrapassagens)"})
comparacao

## Violin Plot das classes

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

metodos = [
    {"titulo": "Original",       "df": df_crystallizer123},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer123_0a10},
    {"titulo": "IQR",            "df": df_crystallizer123_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer123_hampel},
]

fig = plot_violin_classes(
    df_eventos_crystallizer123, metodos, JANELAS,
    titulo="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #1#2#3 unificado"
)
fig.show()

## Plot dados unificados

In [ ]:
# Sem eventos falsos
fig = plot_crystallizer_unificado(
    df_crystallizer123, df_eventos_crystallizer123,
    titulo="Fe (ppm) - Crystallizers Unificados (somente eventos reais)",
    mostrar_falsos=False
)
fig.show()

# Estatísticas descritivas dos eventos

## Crystallizer #1

### Análise por ViolinPlot
Agrupando os dados de todos os eventos por janela e comparando com os dados do falso positivo

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

metodos = [
    {"titulo": "Original",       "df": df_crystallizer1},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer1_0a10},
    {"titulo": "IQR",            "df": df_crystallizer1_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer1_hampel},
]

fig = plot_violin_classes(
    df_eventos_crystallizer1, metodos, JANELAS,
    titulo="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #1"
)
fig.show()

#### Teste estatístico entre as duas classes
Quantificar se diferença entre classes é estatisticamente significativa por janela

In [ ]:
# Dataset: Original — análise de janela de evento roda na Pista A (ver "Política de
# tratamento de dados"): os tratamentos removem justamente max/p90/range, as features
# que separam as classes.
df = df_crystallizer1.copy()
# df = df_crystallizer1_0a10.copy()
# df = df_crystallizer1_iqr.copy()
# df = df_crystallizer1_hampel.copy()

df_testes = testar_classes_por_janela(df, df_eventos_crystallizer1, JANELAS)
df_testes

#### Separabilidade Estatística Feature a Feature por janela
Cálculo do effect size de Mann-Whitney para cada estatística descritiva por janela verificando diretamente qual feature e qual janela têm maior poder discriminativo

In [ ]:
# Dataset: Original — análise de janela de evento roda na Pista A (ver "Política de
# tratamento de dados"): os tratamentos removem justamente max/p90/range, as features
# que separam as classes.
df = df_crystallizer1.copy()
# df = df_crystallizer1_0a10.copy()
# df = df_crystallizer1_iqr.copy()
# df = df_crystallizer1_hampel.copy()

JANELAS = [15, 12, 9, 6, 3]

df_effect, fig = separabilidade_features(
    df, df_eventos_crystallizer1, JANELAS, titulo="Crystallizer #1"
)
fig.show()

### Análise dos eventos ATÉ 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

metodos = [
    {"titulo": "Original",       "df": df_crystallizer1},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer1_0a10},
    {"titulo": "IQR",            "df": df_crystallizer1_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer1_hampel},
]

fig = plot_violin_classes(
    df_eventos_crystallizer1_filtrado_until2020, metodos, JANELAS,
    titulo="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #1 — eventos até 01/04/2020"
)
fig.show()

### Análise dos eventos APÓS 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

metodos = [
    {"titulo": "Original",       "df": df_crystallizer1},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer1_0a10},
    {"titulo": "IQR",            "df": df_crystallizer1_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer1_hampel},
]

fig = plot_violin_classes(
    df_eventos_crystallizer1_filtrado_after2020, metodos, JANELAS,
    titulo="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #1 — eventos após 01/04/2020"
)
fig.show()

## Crystallizer #2

### Análise por ViolinPlot
Agrupando os dados de todos os eventos por janela e comparando com os dados do falso positivo

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

metodos = [
    {"titulo": "Original",       "df": df_crystallizer2},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer2_0a10},
    {"titulo": "IQR",            "df": df_crystallizer2_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer2_hampel},
]

fig = plot_violin_classes(
    df_eventos_crystallizer2, metodos, JANELAS,
    titulo="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #2"
)
fig.show()

#### Teste estatístico entre as duas classes
Quantificar se diferença entre classes é estatisticamente significativa por janela

In [ ]:
# Dataset: Original — análise de janela de evento roda na Pista A (ver "Política de
# tratamento de dados"): os tratamentos removem justamente max/p90/range, as features
# que separam as classes.
df = df_crystallizer2.copy()
# df = df_crystallizer2_0a10.copy()
# df = df_crystallizer2_iqr.copy()
# df = df_crystallizer2_hampel.copy()

df_testes = testar_classes_por_janela(df, df_eventos_crystallizer2, JANELAS)
df_testes

#### Separabilidade Estatística Feature a Feature por janela
Cálculo do effect size de Mann-Whitney para cada estatística descritiva por janela verificando diretamente qual feature e qual janela têm maior poder discriminativo

In [ ]:
# Dataset: Original — análise de janela de evento roda na Pista A (ver "Política de
# tratamento de dados"): os tratamentos removem justamente max/p90/range, as features
# que separam as classes.
df = df_crystallizer2.copy()
# df = df_crystallizer2_0a10.copy()
# df = df_crystallizer2_iqr.copy()
# df = df_crystallizer2_hampel.copy()

JANELAS = [15, 12, 9, 6, 3]

df_effect, fig = separabilidade_features(
    df, df_eventos_crystallizer2, JANELAS, titulo="Crystallizer #2"
)
fig.show()

### Análise dos eventos ATÉ 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

metodos = [
    {"titulo": "Original",       "df": df_crystallizer2},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer2_0a10},
    {"titulo": "IQR",            "df": df_crystallizer2_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer2_hampel},
]

fig = plot_violin_classes(
    df_eventos_crystallizer2_filtrado_until2020, metodos, JANELAS,
    titulo="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #2 — eventos até 01/04/2020"
)
fig.show()

### Análise dos eventos APÓS 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

metodos = [
    {"titulo": "Original",       "df": df_crystallizer2},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer2_0a10},
    {"titulo": "IQR",            "df": df_crystallizer2_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer2_hampel},
]

fig = plot_violin_classes(
    df_eventos_crystallizer2_filtrado_after2020, metodos, JANELAS,
    titulo="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #2 — eventos após 01/04/2020"
)
fig.show()

## Crystallizer #3

### Análise por ViolinPlot
Agrupando os dados de todos os eventos por janela e comparando com os dados do falso positivo

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

metodos = [
    {"titulo": "Original",       "df": df_crystallizer3},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer3_0a10},
    {"titulo": "IQR",            "df": df_crystallizer3_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer3_hampel},
]

fig = plot_violin_classes(
    df_eventos_crystallizer3, metodos, JANELAS,
    titulo="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #3"
)
fig.show()

#### Teste estatístico entre as duas classes
Quantificar se diferença entre classes é estatisticamente significativa por janela

In [ ]:
# Dataset: Original — análise de janela de evento roda na Pista A (ver "Política de
# tratamento de dados"): os tratamentos removem justamente max/p90/range, as features
# que separam as classes.
df = df_crystallizer3.copy()
# df = df_crystallizer3_0a10.copy()
# df = df_crystallizer3_iqr.copy()
# df = df_crystallizer3_hampel.copy()

df_testes = testar_classes_por_janela(df, df_eventos_crystallizer3, JANELAS)
df_testes

#### Separabilidade Estatística Feature a Feature por janela
Cálculo do effect size de Mann-Whitney para cada estatística descritiva por janela verificando diretamente qual feature e qual janela têm maior poder discriminativo

In [ ]:
# Dataset: Original — análise de janela de evento roda na Pista A (ver "Política de
# tratamento de dados"): os tratamentos removem justamente max/p90/range, as features
# que separam as classes.
df = df_crystallizer3.copy()
# df = df_crystallizer3_0a10.copy()
# df = df_crystallizer3_iqr.copy()
# df = df_crystallizer3_hampel.copy()

JANELAS = [15, 12, 9, 6, 3]

df_effect, fig = separabilidade_features(
    df, df_eventos_crystallizer3, JANELAS, titulo="Crystallizer #3"
)
fig.show()

### Análise dos eventos ATÉ 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

metodos = [
    {"titulo": "Original",       "df": df_crystallizer3},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer3_0a10},
    {"titulo": "IQR",            "df": df_crystallizer3_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer3_hampel},
]

fig = plot_violin_classes(
    df_eventos_crystallizer3_filtrado_until2020, metodos, JANELAS,
    titulo="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #3 — eventos até 01/04/2020"
)
fig.show()

### Análise dos eventos APÓS 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

metodos = [
    {"titulo": "Original",       "df": df_crystallizer3},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer3_0a10},
    {"titulo": "IQR",            "df": df_crystallizer3_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer3_hampel},
]

fig = plot_violin_classes(
    df_eventos_crystallizer3_filtrado_after2020, metodos, JANELAS,
    titulo="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #3 — eventos após 01/04/2020"
)
fig.show()

# Comparação dos 3 reatores

## Verificando effect size por reator

In [ ]:
crystallizers_config = [
    # C1
    (df_crystallizer1,        df_eventos_crystallizer1, "C1 Original"),
    (df_crystallizer1_0a10,   df_eventos_crystallizer1, "C1 0-10"),
    (df_crystallizer1_iqr,    df_eventos_crystallizer1, "C1 IQR"),
    (df_crystallizer1_hampel, df_eventos_crystallizer1, "C1 Hampel"),
    # C2
    (df_crystallizer2,        df_eventos_crystallizer2, "C2 Original"),
    (df_crystallizer2_0a10,   df_eventos_crystallizer2, "C2 0-10"),
    (df_crystallizer2_iqr,    df_eventos_crystallizer2, "C2 IQR"),
    (df_crystallizer2_hampel, df_eventos_crystallizer2, "C2 Hampel"),
    # C3
    (df_crystallizer3,        df_eventos_crystallizer3, "C3 Original"),
    (df_crystallizer3_0a10,   df_eventos_crystallizer3, "C3 0-10"),
    (df_crystallizer3_iqr,    df_eventos_crystallizer3, "C3 IQR"),
    (df_crystallizer3_hampel, df_eventos_crystallizer3, "C3 Hampel"),
    # Unificado
    (df_crystallizer123,        df_eventos_crystallizer123, "Unificado Original"),
    (df_crystallizer123_0a10,   df_eventos_crystallizer123, "Unificado 0-10"),
    (df_crystallizer123_iqr,    df_eventos_crystallizer123, "Unificado IQR"),
    (df_crystallizer123_hampel, df_eventos_crystallizer123, "Unificado Hampel"),
]

STATS_FUNCS = {
    'media'   : np.mean,
    'mediana' : np.median,
    'std'     : np.std,
    'max'     : np.max,
    'p75'     : lambda x: np.percentile(x, 75),
    'p90'     : lambda x: np.percentile(x, 90),
    'skewness': lambda x: float(skew(x)),
    'kurtosis': lambda x: float(kurtosis(x)),
    'range'   : lambda x: np.max(x) - np.min(x),
}

registros_todos = []
for df_c, df_ev, nome_c in crystallizers_config:
    for DIAS_JANELA in JANELAS:
        stat_vals = {s: {0: [], 1: []} for s in STATS_FUNCS}
        for _, evento in df_ev.iterrows():
            ts     = evento["TIMESTAMP"]
            inicio = ts - pd.Timedelta(days=DIAS_JANELA)
            mask   = (df_c['TIMESTAMP'] >= inicio) & (df_c['TIMESTAMP'] < ts)
            v      = df_c[mask]["Resultado de Ferro (ppm)"].dropna().values
            if len(v) < 2:
                continue
            classe = int(evento["Real"])
            for nome_stat, func in STATS_FUNCS.items():
                stat_vals[nome_stat][classe].append(func(v))

        for nome_stat in STATS_FUNCS:
            v0 = np.array(stat_vals[nome_stat][0])
            v1 = np.array(stat_vals[nome_stat][1])
            if len(v0) < 2 or len(v1) < 2:
                continue
            stat_mw, p = mannwhitneyu(v0, v1, alternative='two-sided')
            effect = abs(1 - (2 * stat_mw) / (len(v0) * len(v1)))
            registros_todos.append({
                'crystallizer': nome_c,
                'janela'      : f"{DIAS_JANELA}d",
                'feature'     : nome_stat,
                'effect_size' : round(effect, 4),
                'p_value'     : round(p, 4),
            })

df_effect_todos = pd.DataFrame(registros_todos)

# Heatmap — 4 origens (C1, C2, C3, Unificado) × 4 métodos = 16 subplots
n_cols = 4  # Original, 0-10, IQR, Hampel
n_rows = 4  # C1, C2, C3, Unificado
nomes  = [c[2] for c in crystallizers_config]

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=nomes,
    vertical_spacing=0.06
)

for idx, (_, _, nome_c) in enumerate(crystallizers_config):
    row_idx = idx // n_cols + 1
    col_idx = idx % n_cols + 1

    subset = df_effect_todos[df_effect_todos['crystallizer'] == nome_c]
    pivot  = subset.pivot(index='feature', columns='janela', values='effect_size')
    pivot  = pivot[[f"{d}d" for d in JANELAS]]

    fig.add_trace(go.Heatmap(
        z=pivot.values,
        x=pivot.columns.tolist(),
        y=pivot.index.tolist(),
        colorscale='RdYlGn',
        zmin=0, zmax=1,
        text=np.round(pivot.values, 3),
        texttemplate="%{text}",
        showscale=(col_idx == n_cols and row_idx == n_rows)
    ), row=row_idx, col=col_idx)

fig.update_layout(
    title="Effect Size comparativo — C1, C2, C3 e Unificado × Original, 0-10, IQR, Hampel",
    template="plotly_white",
    height=400 * n_rows
)
fig.show()

## Correlação cruzada entre os crystallizers

In [ ]:
freq = "3D"
pares = [("C1", "C2"), ("C1", "C3"), ("C2", "C3")]

metodos = [
    {"titulo": "Original",       "c1": df_crystallizer1,        "c2": df_crystallizer2,        "c3": df_crystallizer3},
    {"titulo": "Intervalo 0-10", "c1": df_crystallizer1_0a10,   "c2": df_crystallizer2_0a10,   "c3": df_crystallizer3_0a10},
    {"titulo": "IQR",            "c1": df_crystallizer1_iqr,    "c2": df_crystallizer2_iqr,    "c3": df_crystallizer3_iqr},
    {"titulo": "Hampel",         "c1": df_crystallizer1_hampel, "c2": df_crystallizer2_hampel, "c3": df_crystallizer3_hampel},
]

for m in metodos:
    s1 = m["c1"].set_index("TIMESTAMP")["Resultado de Ferro (ppm)"].resample(freq).mean()
    s2 = m["c2"].set_index("TIMESTAMP")["Resultado de Ferro (ppm)"].resample(freq).mean()
    s3 = m["c3"].set_index("TIMESTAMP")["Resultado de Ferro (ppm)"].resample(freq).mean()
    m["df_corr"] = pd.concat([s1, s2, s3], axis=1, keys=["C1", "C2", "C3"]).dropna()

    print(f"\n{'='*55}")
    print(f"Método: {m['titulo']}")
    print(m["df_corr"].corr(method="pearson").round(3).to_string())

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=3,
    subplot_titles=[
        f"{a} vs {b} — {m['titulo']}"
        for m in metodos
        for a, b in pares
    ],
    vertical_spacing=0.06
)

for row_idx, m in enumerate(metodos, start=1):
    for col_idx, (a, b) in enumerate(pares, start=1):
        add_scatter_regressao(fig, m["df_corr"][a].values, m["df_corr"][b].values, row=row_idx, col=col_idx)
        fig.update_xaxes(title_text=a, row=row_idx, col=col_idx)
        fig.update_yaxes(title_text=b, row=row_idx, col=col_idx)

fig.update_layout(
    height=400 * n_metodos,
    template='plotly_white',
    title="Correlação cruzada — C1, C2, C3 × Original, 0-10, IQR, Hampel"
)
fig.show()

## Correlação cruzada com lags
Verificando se um reator tem influência sobre outro

In [ ]:
MAX_LAG = 15
lags = range(-MAX_LAG, MAX_LAG + 1)
freq = "3D"
pares = [("C1", "C2"), ("C1", "C3"), ("C2", "C3")]

metodos = [
    {"titulo": "Original",       "c1": df_crystallizer1,        "c2": df_crystallizer2,        "c3": df_crystallizer3},
    {"titulo": "Intervalo 0-10", "c1": df_crystallizer1_0a10,   "c2": df_crystallizer2_0a10,   "c3": df_crystallizer3_0a10},
    {"titulo": "IQR",            "c1": df_crystallizer1_iqr,    "c2": df_crystallizer2_iqr,    "c3": df_crystallizer3_iqr},
    {"titulo": "Hampel",         "c1": df_crystallizer1_hampel, "c2": df_crystallizer2_hampel, "c3": df_crystallizer3_hampel},
]

# Prepara df_corr para cada método
for m in metodos:
    s1 = m["c1"].set_index("TIMESTAMP")["Resultado de Ferro (ppm)"].resample(freq).mean()
    s2 = m["c2"].set_index("TIMESTAMP")["Resultado de Ferro (ppm)"].resample(freq).mean()
    s3 = m["c3"].set_index("TIMESTAMP")["Resultado de Ferro (ppm)"].resample(freq).mean()
    m["df_corr"] = pd.concat([s1, s2, s3], axis=1, keys=["C1", "C2", "C3"]).dropna()

# Cross-correlação com lag
n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Cross-correlação com Defasagem — {m['titulo']}" for m in metodos],
    vertical_spacing=0.08
)

for row_idx, m in enumerate(metodos, start=1):
    df_c = m["df_corr"]
    resultados_lag = []

    for a, b in pares:
        x = df_c[a].values
        y = df_c[b].values
        for lag in lags:
            if lag < 0:
                xs, ys = x[:lag],  y[-lag:]
            elif lag > 0:
                xs, ys = x[lag:],  y[:-lag]
            else:
                xs, ys = x, y
            r, p = pearsonr(xs, ys)
            resultados_lag.append({
                "par": f"{a} vs {b}", "lag_dias": lag * 3,
                "r": round(r, 4), "p": round(p, 6)
            })

    df_lag = pd.DataFrame(resultados_lag)

    for par in df_lag["par"].unique():
        sub = df_lag[df_lag["par"] == par]
        fig.add_trace(go.Scatter(
            x=sub["lag_dias"], y=sub["r"],
            mode="lines", name=par,
            line=dict(width=2),
            legendgroup=par,
            showlegend=(row_idx == 1)
        ), row=row_idx, col=1)

    fig.add_vline(x=0, line_dash="dash", line_color="black")
    fig.add_hline(y=0, line_color="gray", line_width=0.5, row=row_idx, col=1)
    fig.update_yaxes(title_text="Pearson r", row=row_idx, col=1)
    fig.update_xaxes(title_text="Defasagem (dias)", row=row_idx, col=1)

    # Lag de máxima correlação
    print(f"\nMétodo: {m['titulo']} — Lag de máxima correlação:")
    for par in df_lag["par"].unique():
        sub  = df_lag[df_lag["par"] == par]
        best = sub.loc[sub["r"].idxmax()]
        print(f"  {par}: lag={best['lag_dias']:.0f} dias  |  r={best['r']:.4f}")

fig.update_layout(
    height=400 * n_metodos,
    template="plotly_white",
    hovermode="x unified",
    title="Correlação cruzada com Defasagem entre Reatores<br>"
          "<sup>Pico em lag≠0 indica que um reator influencia o outro, negativo: A influencia B  |  positivo: B influencia A</sup>"
)
fig.show()

# Clusterização (Não supervisionada)

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

STATS_FUNCS_CLUSTERING = {
    'media':   np.mean,
    'mediana': np.median,
    'std':     np.std,
    'max':     np.max,
    'p75':     lambda x: np.percentile(x, 75),
    'p90':     lambda x: np.percentile(x, 90),
    'range':   lambda x: np.max(x) - np.min(x),
}

# Pista A (ver "Política de tratamento de dados"): modelos rodam só sobre o Original.
# Os tratamentos foram retirados daqui de propósito — Intervalo 0-10, IQR e Hampel removem
# as leituras-gatilho (264/999 ppm) e zeram max/p90/range nas janelas de evento, então
# comparar modelos entre tratamentos só gasta o pouco poder estatístico dos 31 positivos.
configs_crystallizer = {
    "C1":   {"Original": (df_crystallizer1,   df_eventos_crystallizer1)},
    "C2":   {"Original": (df_crystallizer2,   df_eventos_crystallizer2)},
    "C3":   {"Original": (df_crystallizer3,   df_eventos_crystallizer3)},
    "C123": {"Original": (df_crystallizer123, df_eventos_crystallizer123)},
}

## Crystallizer #1

In [ ]:
CRYSTALLIZER = "C1"  # "C1" | "C2" | "C3" | "C123"

resultados_por_dataset = rodar_clusterizacao(
    CRYSTALLIZER, configs_crystallizer[CRYSTALLIZER], JANELAS, STATS_FUNCS_CLUSTERING
)

In [ ]:
fig = plotar_clusterizacao_pca(resultados_por_dataset, CRYSTALLIZER)
fig.show()

## Crystallizer #2

In [ ]:
CRYSTALLIZER = "C2"  # "C1" | "C2" | "C3" | "C123"

resultados_por_dataset = rodar_clusterizacao(
    CRYSTALLIZER, configs_crystallizer[CRYSTALLIZER], JANELAS, STATS_FUNCS_CLUSTERING
)

In [ ]:
fig = plotar_clusterizacao_pca(resultados_por_dataset, CRYSTALLIZER)
fig.show()

## Crystallizer #3

In [ ]:
CRYSTALLIZER = "C3"  # "C1" | "C2" | "C3" | "C123"

resultados_por_dataset = rodar_clusterizacao(
    CRYSTALLIZER, configs_crystallizer[CRYSTALLIZER], JANELAS, STATS_FUNCS_CLUSTERING
)

In [ ]:
fig = plotar_clusterizacao_pca(resultados_por_dataset, CRYSTALLIZER)
fig.show()

## Crystallizer #1 #2 #3

In [ ]:
CRYSTALLIZER = "C123"  # "C1" | "C2" | "C3" | "C123"

resultados_por_dataset = rodar_clusterizacao(
    CRYSTALLIZER, configs_crystallizer[CRYSTALLIZER], JANELAS, STATS_FUNCS_CLUSTERING
)

In [ ]:
fig = plotar_clusterizacao_pca(resultados_por_dataset, CRYSTALLIZER)
fig.show()

# Classificação (Abordagem Supervisionada)

In [ ]:
JANELAS       = [15, 12, 9, 6, 3]
N_SPLITS      = 5
RANDOM_STATE  = 42
TEST_SIZE     = 0.2

# Sobrescreve o STATS_FUNCS padrão do utils.py para as células desta seção
STATS_FUNCS = {
    'media':   np.mean,
    'mediana': np.median,
    'std':     np.std,
    'max':     np.max,
    'p75':     lambda x: np.percentile(x, 75),
    'p90':     lambda x: np.percentile(x, 90),
    'range':   lambda x: np.max(x) - np.min(x),
}

# Pista A (ver "Política de tratamento de dados"): treino e avaliação SEMPRE sobre o
# Original (bruto + limpar_excursoes). Os tratamentos deletam as leituras-gatilho e
# invalidam a comparação com a regra vigente (no IQR nada passa de 4.0 ppm — a regra
# de 5 ppm nunca dispararia). Robustez vem das features (mediana, p75/p90, relativas),
# não de pré-filtro.
MAP_MEDICOES = {
    "Original": {"C1": df_crystallizer1, "C2": df_crystallizer2, "C3": df_crystallizer3},
}

CONFIGS = {
    "C1": {"Original": (df_crystallizer1, df_eventos_crystallizer1)},
    "C2": {"Original": (df_crystallizer2, df_eventos_crystallizer2)},
    "C3": {"Original": (df_crystallizer3, df_eventos_crystallizer3)},
}

## Crystallizer #1

In [ ]:
MODO = "C1"
TRATAMENTO_ALVO = "Original"  # Pista A: modelos rodam sobre o Original (política de tratamento)

df_med, df_ev = CONFIGS[MODO][TRATAMENTO_ALVO]

df_relatorio, resultados_para_plot = rodar_classificacao(
    MODO, df_med, df_ev, JANELAS, tratamento_alvo=TRATAMENTO_ALVO, stats_funcs=STATS_FUNCS
)

## Crystallizer #2

In [ ]:
MODO = "C2"
TRATAMENTO_ALVO = "Original"  # Pista A: modelos rodam sobre o Original (política de tratamento)

df_med, df_ev = CONFIGS[MODO][TRATAMENTO_ALVO]

df_relatorio, resultados_para_plot = rodar_classificacao(
    MODO, df_med, df_ev, JANELAS, tratamento_alvo=TRATAMENTO_ALVO, stats_funcs=STATS_FUNCS
)

## Crystallizer #3

In [ ]:
MODO = "C3"
TRATAMENTO_ALVO = "Original"  # Pista A: modelos rodam sobre o Original (política de tratamento)

df_med, df_ev = CONFIGS[MODO][TRATAMENTO_ALVO]

df_relatorio, resultados_para_plot = rodar_classificacao(
    MODO, df_med, df_ev, JANELAS, tratamento_alvo=TRATAMENTO_ALVO, stats_funcs=STATS_FUNCS
)

## Crystallizer #1 #2 #3

In [ ]:
MODO = "Unificado"

# Pista A: só o Original entra nos modelos (ver "Política de tratamento de dados")
CONFIGS_UNIFICADO = {
    "Original": df_crystallizer123,
}

resultados_por_tratamento_unif, df_cv_consolidado_unif = rodar_classificacao_por_tratamento(
    MODO, CONFIGS_UNIFICADO, df_eventos_crystallizer123, JANELAS,
    stats_funcs=STATS_FUNCS, test_size=TEST_SIZE, usar_smote=False
)

# IsolationForest

In [ ]:
JANELAS = [6, 3, 1]
N_SPLITS     = 5
RANDOM_STATE = 42
TEST_SIZE    = 0.1   # 20% para teste
VAL_SIZE     = 0.2   # 20% do treino para validação (dentro do CV)

# Sobrescreve o STATS_FUNCS padrão do utils.py para as células desta seção
STATS_FUNCS = {
    'media':   np.mean,
    'mediana': np.median,
    'std':     np.std,
    'max':     np.max
}

# Definindo o grid de parâmetros para testar
PARAM_GRID = {
    'contamination': [0.01, 0.03, 0.05, 0.10],
    'n_estimators': [100, 300, 500],
    'max_features': [0.5, 0.8, 1.0],
    'max_samples': ['auto', 0.5, 0.8]
}

## Crystallizer #1

In [ ]:
MODO = "C1"

resultados_iforest = rodar_iforest(MODO, CONFIGS[MODO], JANELAS, PARAM_GRID, stats_funcs=STATS_FUNCS)

## Crystallizer #2

In [ ]:
MODO = "C2"

resultados_iforest = rodar_iforest(MODO, CONFIGS[MODO], JANELAS, PARAM_GRID, stats_funcs=STATS_FUNCS)

## Crystallizer #3

In [ ]:
MODO = "C3"

resultados_iforest = rodar_iforest(MODO, CONFIGS[MODO], JANELAS, PARAM_GRID, stats_funcs=STATS_FUNCS)

## Crystallizer #1#2#3

In [ ]:
MODO = "Unificado"

# No unificado a base de medições vem do dicionário e a tabela de eventos é fixa
configs_unificado = {
    tratamento: (df_med, df_eventos_crystallizer123)
    for tratamento, df_med in CONFIGS_UNIFICADO.items()
}

resultados_iforest_unif = rodar_iforest(MODO, configs_unificado, JANELAS, PARAM_GRID, stats_funcs=STATS_FUNCS)

# Carta de Controle
Avaliando carta de controle EWMA verificando se há antecedência nos eventos de falha no reator

## EWMA

In [ ]:
# RESSALVA (documentação de decisão): o baseline abaixo é escolhido à mão em períodos de
# 2012-2015. A série tem drift descendente forte (mediana anual ~2.7 ppm -> ~1.8 ppm após
# 2020), então limites calculados nesse baseline ficam sistematicamente frouxos no regime
# atual — e o grid com L até 13 tende a compensar isso ajustando ruído. Próxima etapa:
# baseline rolante (como o CUSUM dinâmico abaixo) ou recálculo por regime (corte 01/04/2020).
df_c1 = df_crystallizer1.copy()
df_c1['TIMESTAMP'] = pd.to_datetime(df_c1['TIMESTAMP'])
df_c1 = df_c1.sort_values('TIMESTAMP').set_index('TIMESTAMP')
serie_fe = df_c1['Resultado de Ferro (ppm)'].dropna()

df_eventos_c1 = df_eventos_crystallizer1.copy()
df_eventos_c1['TIMESTAMP'] = pd.to_datetime(df_eventos_c1['TIMESTAMP'])

periodos_baseline = [
    ('2012-03-01', '2013-03-25'),
    ('2014-01-01', '2015-01-01'),
    # ('2022-05-01', '2023-10-01')
]
fatias = [serie_fe.loc[inicio:fim] for inicio, fim in periodos_baseline]
serie_baseline = pd.concat(fatias)

media_historica = serie_baseline.mean()
std_historico   = serie_baseline.std()

print(f"Baseline | Média: {media_historica:.3f} ppm | Std: {std_historico:.3f} ppm")
print(f"Total de amostras no baseline: {len(serie_baseline)}\n")

# 1. Defina as opções matemáticas que você quer testar
param_grid_ewma = {
    'lambd': [0.1, 0.2, 0.3, 0.4, 0.5],       # O peso dos dados recentes
    'L': [3, 5, 7, 9, 11, 13],                # O multiplicador do limite
    'janela_dias': [3, 6, 9, 12, 15]          # Antecedência do alarme
}

# 2. Roda a otimização
melhores_parametros, max_f2 = otimizar_ewma(
    serie_fe, df_eventos_c1, media_historica, std_historico, param_grid_ewma
)

# 3. Gera a carta final com a melhor configuração encontrada
df_ewma_otimizado = calcular_ewma(
    serie_fe, media_historica, std_historico, 
    lambd=melhores_parametros['lambd'], 
    L=melhores_parametros['L']
)

# 4. Avalia formalmente para imprimir o relatório completo
avaliar_carta_controle(
    df_ewma_otimizado, df_eventos_c1, 
    nome_carta=f"EWMA Otimizado (λ={melhores_parametros['lambd']}, L={melhores_parametros['L']})", 
    janela_dias=melhores_parametros['janela_dias']
)

# 5. Plota o gráfico final
plotar_carta_ewma(
    df_ewma_otimizado, df_eventos_c1,
    titulo=f"Carta EWMA - C1 (λ={melhores_parametros['lambd']}, L={melhores_parametros['L']})",
    mostrar_falsos=False
)

## CUMSUM

In [ ]:
# ==========================================
# 2. FUNÇÃO DE PLOTAGEM
# ==========================================

# ==========================================
# 3. BLOCO DE EXECUÇÃO E OTIMIZAÇÃO
# ==========================================

# Preparação dos dados
df_c1 = df_crystallizer1.copy()
df_c1['TIMESTAMP'] = pd.to_datetime(df_c1['TIMESTAMP'])
df_c1 = df_c1.sort_values('TIMESTAMP').set_index('TIMESTAMP')
serie_fe = df_c1['Resultado de Ferro (ppm)'].dropna()

df_eventos_c1 = df_eventos_crystallizer1.copy()
df_eventos_c1['TIMESTAMP'] = pd.to_datetime(df_eventos_c1['TIMESTAMP'])

# Definição do Grid de Parâmetros para buscar a melhor performance
param_grid_cusum = {
    'janela_baseline': [15, 30, 45, 60], # Quantos dias passados compõem o "normal"
    'k': [0.25, 0.5, 0.75],              # Folga (menor = soma desvios menores)
    'h': [3, 4, 5, 6],                   # Limite de alarme (maior = mais rigoroso)
    'janela_dias': [3, 6, 9, 12]         # Janela de antecedência para prever a falha
}

# Roda a otimização
melhores_parametros, max_f2 = otimizar_cusum_dinamico(serie_fe, df_eventos_c1, param_grid_cusum)

# Gera a carta final com os hiperparâmetros campeões
df_cusum_otimizado = calcular_cusum_dinamico(
    serie_fe, 
    janela_baseline=melhores_parametros['janela_baseline'], 
    k=melhores_parametros['k'], 
    h=melhores_parametros['h']
)

# Avalia formalmente para exibir o relatório
nome_modelo = f"CUSUM Dinâmico (jan_base={melhores_parametros['janela_baseline']}, k={melhores_parametros['k']}, h={melhores_parametros['h']})"
avaliar_carta_controle(df_cusum_otimizado, df_eventos_c1, nome_carta=nome_modelo, janela_dias=melhores_parametros['janela_dias'])

# Plota o gráfico para análise visual
plotar_carta_cusum(
    df_cusum_otimizado, df_eventos_c1,
    titulo=f"Carta CUSUM Dinâmico - C1 {nome_modelo}",
    mostrar_falsos=False
)